In [ ]:
"""
SimCLR 质谱二分类微调入口（ main.ipynb ）
- 核心逻辑已剥离至 lib/ 目录下以提高可维护性
- 本文件只保留运行入口、配置项与流程编排
"""

import sys
import torch
from pathlib import Path
from datetime import datetime

# ==================== 1. 环境与路径配置 ====================
base_dir = Path(__file__).parent if '__file__' in locals() else Path.cwd()
project_root = base_dir.parent if base_dir.name == 'simclr_finetune' else base_dir

# 确保项目根目录在 python 模块搜索路径中
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 导入 simclr_finetune 模块
from simclr_finetune.lib import (
    load_pretrained_encoder,
    save_model_checkpoint,
    BinaryClassifier,
    prepare_finetune_dataset,
    train_binary_classifier,
    evaluate_model,
    evaluate_and_record_predictions,
    evaluate_positive_per_smiles,
    export_results_to_excel,
    plot_comprehensive_results,
    plot_confusion_matrices
)

# 超参数配置
batch_size = 128
lr = 0.001
epochs = 100
patience = 15
freeze_encoder = True
pos_weight = 1.55
threshold = 0.9  # 推理判定阈值调整为 0.9

# 设置推理与微调设备 (优先 CUDA)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("=" * 60)
print("SimCLR 质谱二分类微调管线")
print("=" * 60)
print(f"运行设备: {device.upper()}")
if device == 'cuda':
    print(f"显卡型号: {torch.cuda.get_device_name(0)}")
print(f"推理判定阈值: {threshold}")

# 数据与模型文件路径
pos_msp_path = base_dir / "data_source" / "阳性-含CanonicalSMILES-5类骨架(3).msp"
neg_msp_path = base_dir / "data_source" / "阴性(4).msp"

# 备选退避路径
if not pos_msp_path.exists():
    pos_msp_path = project_root / "known_compound_recognize" / "positive.msp"
if not neg_msp_path.exists():
    neg_msp_path = project_root / "scratch" / "阴性.msp"

encoder_path = project_root / "simclr_pretrain" / "best_model.pt"
if not encoder_path.exists():
    encoder_path = project_root / "simclr_pretrain" / "pretrained_encoder_final.pt"

output_dir = base_dir
output_dir.mkdir(parents=True, exist_ok=True)

print(f"正样本路径: {pos_msp_path}")
print(f"负样本路径: {neg_msp_path}")
print(f"编码器路径: {encoder_path}")

# ==================== 2. 数据准备与划分 ====================
print("\n" + "=" * 60)
print("Step 1: 数据加载、预处理与划分")
print("=" * 60)
train_loader, train_eval_loader, val_loader, test_loader, meta = prepare_finetune_dataset(
    pos_msp_path=pos_msp_path,
    neg_msp_path=neg_msp_path,
    batch_size=batch_size,
    test_size=0.15,
    val_size=0.15,
    random_state=42,
    use_cuda=(device == 'cuda')
)

print(f"  训练集批次数: {len(train_loader)}")
print(f"  验证集批次数: {len(val_loader)}")
print(f"  测试集批次数: {len(test_loader)}")

# ==================== 3. 模型构建与权重加载 ====================
print("\n" + "=" * 60)
print("Step 2: 构建模型并加载预训练编码器")
print("=" * 60)
encoder = load_pretrained_encoder(
    encoder_path=encoder_path,
    input_dim=561,
    hidden_dim=256,
    device=device
)
model = BinaryClassifier(encoder, input_dim=256, freeze_encoder=freeze_encoder)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  可训练参数量: {trainable_params:,}")

# ==================== 4. 微调训练 ====================
print("\n" + "=" * 60)
print("Step 3: 开始二分类微调训练")
print("=" * 60)
start_time = datetime.now()
model, history = train_binary_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=epochs,
    lr=lr,
    patience=patience,
    pos_weight=pos_weight
)
elapsed = datetime.now() - start_time
print(f"  [OK] 训练完成，总用时: {str(elapsed).split('.')[0]}")

# ==================== 5. 综合评估 ====================
print("\n" + "=" * 60)
print(f"Step 4: 模型性能评估 (判定阈值={threshold})")
print("=" * 60)
train_res = evaluate_and_record_predictions(model, train_eval_loader, sample_names=meta['train_sample_names'], device=device, threshold=threshold)
val_res = evaluate_and_record_predictions(model, val_loader, sample_names=meta['val_sample_names'], device=device, threshold=threshold)
test_res = evaluate_and_record_predictions(model, test_loader, sample_names=meta['test_sample_names'], device=device, threshold=threshold)

print(f"  训练集 | Accuracy: {train_res['accuracy']:.2%} | AUC: {train_res['auc']:.4f}")
print(f"  验证集 | Accuracy: {val_res['accuracy']:.2%} | AUC: {val_res['auc']:.4f}")
print(f"  测试集 | Accuracy: {test_res['accuracy']:.2%} | AUC: {test_res['auc']:.4f}")

smiles_res = evaluate_positive_per_smiles(
    test_indices=meta['test_idx'],
    smiles_all=meta['smiles_all'],
    labels_all=meta['y'],
    probs=test_res['probs'],
    preds=test_res['preds'],
    threshold=threshold
)

# ==================== 6. 结果保存与导出 ====================
print("\n" + "=" * 60)
print("Step 5: 保存模型与导出评估结果")
print("=" * 60)
# 保存为全新时间戳权重文件
saved_path = save_model_checkpoint({
    'encoder_state_dict': encoder.state_dict(),
    'classifier_state_dict': model.classifier.state_dict(),
    'history': history,
    'test_metrics': test_res,
}, output_dir)

# 导出 Excel / CSV 与 绘图
export_results_to_excel(
    history=history,
    model=model,
    train_results=train_res,
    val_results=val_res,
    test_results=test_res,
    smiles_results=smiles_res,
    meta=meta,
    output_dir=output_dir
)
plot_comprehensive_results(history, train_res, val_res, test_res, smiles_res, output_dir)
plot_confusion_matrices(train_res, val_res, test_res, output_dir)

print("\n" + "=" * 60)
print("微调管线全部顺利完成！")
print("=" * 60)
